In [30]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [31]:
load_dotenv()

True

In [32]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "conversational"
 )

model = ChatHuggingFace(llm = llm)

In [33]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [34]:
def gen_joke(state:JokeState):
    prompt = f"generate a joke on the topic {state['topic']}"
    response = model.invoke(prompt).content
    return {'joke':response}

In [35]:
def gen_explaination(state:JokeState):
    prompt = f"generate explaination of the given joke {state['joke']}"
    response = model.invoke(prompt).content
    return {'explaination':response}

In [36]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',gen_joke)
graph.add_node('generate_explaination',gen_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

checkpoint = InMemorySaver()

workflow = graph.compile(checkpointer = checkpoint)


In [37]:
config1 = {'configurable': {'thread_id': 'qwwen1random'}}
workflow.invoke({'topic': 'pizza'},config = config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a passport?\n\nBecause it wanted to go on a slice-erland vacation!',
 'explaination': 'This joke plays on wordplay and puns. Specifically:\n\n1. **Homophonic Wordplay**: The word "slice" in the joke is a play on the word "slič", which sounds similar to "slice" but means "pretty" or "attractive" in some Slavic languages, like Croatian. In this context, it\'s turned into "slič," which can be interpreted as "slice-erland," suggesting a fictional place for pizza slices to visit.\n\n2. **Pun**: The joke uses the word "slice" in a double meaning. It refers to a part of a pizza, and also to the word "slič," which is a playful creation. The reference to "slič" or "slice-erland" is a humorous twist on the idea of a pizza traveling.\n\n3. **Vacation**: The punchline suggests that the pizza wants to go on a "vacation," which is a common activity for people but a somewhat humorous concept for an inanimate object like a pizza.\n\nSo, the humo

In [38]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a passport?\n\nBecause it wanted to go on a slice-erland vacation!', 'explaination': 'This joke plays on wordplay and puns. Specifically:\n\n1. **Homophonic Wordplay**: The word "slice" in the joke is a play on the word "slič", which sounds similar to "slice" but means "pretty" or "attractive" in some Slavic languages, like Croatian. In this context, it\'s turned into "slič," which can be interpreted as "slice-erland," suggesting a fictional place for pizza slices to visit.\n\n2. **Pun**: The joke uses the word "slice" in a double meaning. It refers to a part of a pizza, and also to the word "slič," which is a playful creation. The reference to "slič" or "slice-erland" is a humorous twist on the idea of a pizza traveling.\n\n3. **Vacation**: The punchline suggests that the pizza wants to go on a "vacation," which is a common activity for people but a somewhat humorous concept for an inanimate object like a piz

In [41]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a passport?\n\nBecause it wanted to go on a slice-erland vacation!', 'explaination': 'This joke plays on wordplay and puns. Specifically:\n\n1. **Homophonic Wordplay**: The word "slice" in the joke is a play on the word "slič", which sounds similar to "slice" but means "pretty" or "attractive" in some Slavic languages, like Croatian. In this context, it\'s turned into "slič," which can be interpreted as "slice-erland," suggesting a fictional place for pizza slices to visit.\n\n2. **Pun**: The joke uses the word "slice" in a double meaning. It refers to a part of a pizza, and also to the word "slič," which is a playful creation. The reference to "slič" or "slice-erland" is a humorous twist on the idea of a pizza traveling.\n\n3. **Vacation**: The punchline suggests that the pizza wants to go on a "vacation," which is a common activity for people but a somewhat humorous concept for an inanimate object like a pi

In [42]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!',
 'explaination': 'This joke plays on a play of words and a play on a cooking term. The phrase "al dente" is a culinary term used to describe pasta that is cooked to a just-right texture, meaning it\'s firm to the bite. The joke takes this term and turns it into a pun by implying that the pasta is feeling a bit "al dente" as in feeling a bit tense or nervous, as one might feel when going to the doctor. This dual meaning makes the joke humorous, as it combines a food-related term with a more common usage in a lighthearted way.'}

In [43]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!', 'explaination': 'This joke plays on a play of words and a play on a cooking term. The phrase "al dente" is a culinary term used to describe pasta that is cooked to a just-right texture, meaning it\'s firm to the bite. The joke takes this term and turns it into a pun by implying that the pasta is feeling a bit "al dente" as in feeling a bit tense or nervous, as one might feel when going to the doctor. This dual meaning makes the joke humorous, as it combines a food-related term with a more common usage in a lighthearted way.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fcce-c7af-6293-8002-9a5c85e46f0d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-04T04:21:57.064330+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fcce-ba5

In [47]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!', 'explaination': 'This joke plays on a play of words and a play on a cooking term. The phrase "al dente" is a culinary term used to describe pasta that is cooked to a just-right texture, meaning it\'s firm to the bite. The joke takes this term and turns it into a pun by implying that the pasta is feeling a bit "al dente" as in feeling a bit tense or nervous, as one might feel when going to the doctor. This dual meaning makes the joke humorous, as it combines a food-related term with a more common usage in a lighthearted way.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fcce-c7af-6293-8002-9a5c85e46f0d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-04T04:21:57.064330+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fcce-ba

In [ ]:
#TIME TRAVEL

In [48]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id":"1f15fcce-ad8b-6d56-8000-73263790ce78"}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f15fcce-ad8b-6d56-8000-73263790ce78'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-04T04:21:54.323577+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fcce-ad85-608d-bfff-e2b89d7a858e'}}, tasks=(PregelTask(id='145692df-71e6-2e3f-2e9e-2e390a063611', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!'}),), interrupts=())

In [51]:
workflow.invoke(None, {"configurable": {"thread_id": "qwwen1random", "checkpoint_id": "1f15fc08-dfea-64f4-8002-51008b325268"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a passport?\n\nBecause it wanted to go on a slice-erland vacation!',
 'explaination': 'This joke plays on wordplay and puns. Specifically:\n\n1. **Homophonic Wordplay**: The word "slice" in the joke is a play on the word "slič", which sounds similar to "slice" but means "pretty" or "attractive" in some Slavic languages, like Croatian. In this context, it\'s turned into "slič," which can be interpreted as "slice-erland," suggesting a fictional place for pizza slices to visit.\n\n2. **Pun**: The joke uses the word "slice" in a double meaning. It refers to a part of a pizza, and also to the word "slič," which is a playful creation. The reference to "slič" or "slice-erland" is a humorous twist on the idea of a pizza traveling.\n\n3. **Vacation**: The punchline suggests that the pizza wants to go on a "vacation," which is a common activity for people but a somewhat humorous concept for an inanimate object like a pizza.\n\nSo, the humo

In [53]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a passport?\n\nBecause it wanted to go on a slice-erland vacation!', 'explaination': 'This joke plays on wordplay and puns. Specifically:\n\n1. **Homophonic Wordplay**: The word "slice" in the joke is a play on the word "slič", which sounds similar to "slice" but means "pretty" or "attractive" in some Slavic languages, like Croatian. In this context, it\'s turned into "slič," which can be interpreted as "slice-erland," suggesting a fictional place for pizza slices to visit.\n\n2. **Pun**: The joke uses the word "slice" in a double meaning. It refers to a part of a pizza, and also to the word "slič," which is a playful creation. The reference to "slič" or "slice-erland" is a humorous twist on the idea of a pizza traveling.\n\n3. **Vacation**: The punchline suggests that the pizza wants to go on a "vacation," which is a common activity for people but a somewhat humorous concept for an inanimate object like a pi

In [ ]:
#UPDATING STATE

In [62]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id":"1f15fcce-ba50-6bd9-8001-82028c18aae0", "checkpoint_ns": ""}},{'topic':'sambar'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f15fd63-9889-6c22-8002-a33ae9b477f4'}}

In [63]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'sambar', 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!'}, next=('generate_explaination',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fd63-9889-6c22-8002-a33ae9b477f4'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-06-04T05:28:31.808995+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15fcce-ba50-6bd9-8001-82028c18aae0'}}, tasks=(PregelTask(id='51624c41-6406-590f-e1ae-c901fcf7dd2b', name='generate_explaination', path=('__pregel_pull', 'generate_explaination'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'sambar', 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!', 'explaination': 'This joke is a play on words and a pun. The phrase "al dente" is a culinary term used to describe pasta that is c

In [65]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f15fd63-9889-6c22-8002-a33ae9b477f4"}})

{'topic': 'sambar',
 'joke': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little al dente!',
 'explaination': 'This joke plays on the word "al dente," which is a term used in cooking to describe pasta that is perfectly cooked and still has a little firmness to it. In the joke, the word "al dente" is used to sound like "a little dent," suggesting that the pasta is feeling a bit unwell, much like a person might feel if they were visiting a doctor. The humor comes from the pun and the unexpected comparison between something typically associated with food preparation and a medical visit.'}

In [ ]:
list(workflow.get_state_history(config2))